In [ ]:
import pandas as pd
import gzip
import urllib.request
import os

def get_uniprot_mapping(
    url: str, 
    file_path: str, 
    target_type: str = 'Gene_Name', 
    output_col_name: str = 'gene_symbol',
    tax_id: int = None,
    organism_name: str = None
) -> pd.DataFrame:
    """
    UniProt ID 매핑 데이터를 다운로드하고 특정 ID 타입만 필터링하여 반환합니다.
    
    Args:
        url (str): 데이터 다운로드 URL
        file_path (str): 저장할 로컬 파일 경로 (gz 포함)
        target_type (str): 필터링할 ID 타입 (예: 'Gene_Name', 'Ensembl' 등)
        output_col_name (str): 결과 데이터프레임의 타겟 컬럼명 변경값
        tax_id (int): Taxonomy ID (예: 9606, 10090)
        organism_name (str): 생물종 이름 (예: 'Homo sapiens', 'Mus musculus')
        
    Returns:
        pd.DataFrame: ['uniprot_id', output_col_name, 'tax_id', 'organism'] 컬럼을 가진 데이터프레임
    """
    
    # 1. 파일 다운로드 (없을 경우에만)
    if not os.path.exists(file_path):
        print(f"📥 '{file_path}' 다운로드 중... (시간이 소요됩니다)")
        try:
            urllib.request.urlretrieve(url, file_path)
            print("✅ 다운로드 완료.")
        except Exception as e:
            print(f"❌ 다운로드 실패: {e}")
            return pd.DataFrame()
    else:
        print(f"ℹ️ 기존 파일 '{file_path}'을(를) 사용합니다.")

    # 2. 데이터 로드 및 필터링
    print(f"🔄 '{target_type}' 데이터 처리 중...")
    chunk_size = 100_000
    results = []

    try:
        with gzip.open(file_path, 'rt') as f:
            # 필요한 컬럼만 지정하여 읽기
            reader = pd.read_csv(
                f, 
                sep='\t', 
                header=None, 
                names=['UniProtID', 'Type', 'Value'], 
                chunksize=chunk_size
            )
            
            for chunk in reader:
                # 조건에 맞는 행만 추출 (메모리 절약)
                subset = chunk.loc[chunk['Type'] == target_type, ['UniProtID', 'Value']]
                results.append(subset)
                
    except Exception as e:
        print(f"파일 읽기 오류: {e}")
        return pd.DataFrame()

    # 3. 결과 병합 및 정리
    if results:
        df_final = pd.concat(results, ignore_index=True)
        
        # 컬럼 이름 변경 (사용자가 지정한 이름으로)
        df_final.columns = ['uniprot_id', output_col_name]
        
        # 중복 제거
        df_final.drop_duplicates(inplace=True)
        
        # tax_id와 organism 컬럼 추가
        if tax_id is not None:
            df_final['tax_id'] = tax_id
        if organism_name is not None:
            df_final['organism'] = organism_name
        
        print(f"✅ 처리 완료: {len(df_final)}개 매핑 발견")
        return df_final
    else:
        print("⚠️ 해당 타입의 데이터가 없습니다.")
        return pd.DataFrame()

# ==========================================
# 실행: Human과 Mouse 데이터 다운로드 및 처리
# ==========================================

# 경로 설정
RAW_DIR = "/data/project/shinyoung/DB/raw/uniprot"  # raw 데이터 저장
PROCESSED_DIR = "/data/project/shinyoung/DB/processed/Uniprot"  # 처리된 데이터 저장

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Human 데이터
HUMAN_URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz"
HUMAN_FILE = f"{RAW_DIR}/HUMAN_9606_idmapping.dat.gz"

# Mouse 데이터
MOUSE_URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/MOUSE_10090_idmapping.dat.gz"
MOUSE_FILE = f"{RAW_DIR}/MOUSE_10090_idmapping.dat.gz"

print("="*60)
print("Human 데이터 처리")
print("="*60)
df_human = get_uniprot_mapping(
    HUMAN_URL, HUMAN_FILE, 
    target_type='Gene_Name', 
    output_col_name='gene_symbol',
    tax_id=9606,
    organism_name='Homo sapiens'
)
print(f"\n[Human 데이터 미리보기]")
print(df_human.head())

print("\n" + "="*60)
print("Mouse 데이터 처리")
print("="*60)
df_mouse = get_uniprot_mapping(
    MOUSE_URL, MOUSE_FILE, 
    target_type='Gene_Name', 
    output_col_name='gene_symbol',
    tax_id=10090,
    organism_name='Mus musculus'
)
print(f"\n[Mouse 데이터 미리보기]")
print(df_mouse.head())

# ==========================================
# 데이터 통합 (Human + Mouse)
# ==========================================
print("\n" + "="*60)
print("데이터 통합 중...")
print("="*60)

df_integrated = pd.concat([df_human, df_mouse], ignore_index=True)

print(f"✅ 통합 완료!")
print(f"총 행 수: {len(df_integrated):,}개")
print(f"컬럼 목록: {list(df_integrated.columns)}")

print("\n[종별 데이터 분포]")
print(df_integrated['organism'].value_counts())

# 파일 저장
OUTPUT_FILE = f"{PROCESSED_DIR}/uniprot_data_mouse_human.csv"
df_integrated.to_csv(OUTPUT_FILE, index=False)
print(f"\n💾 저장 완료: {OUTPUT_FILE}")

print("\n[통합 데이터 미리보기]")
print(df_integrated.head())
print("...")
print(df_integrated.tail())

In [3]:
df_final.to_csv("uniprot-data/uniprot_gene_mapping_human.csv", index=False)
df_gene.to_csv("uniprot-data/uniprot_gene_mapping_mouse.csv", index=False)

In [10]:
df_mouse

,UniProtID,Type,Value
1,Q60888,Gene_Name,Or10d1b
46,Q8VBW9,Gene_Name,Or10h28
85,Q60885,Gene_Name,Or10p22
122,Q9CQV8,Gene_Name,Ywhab
209,P62259,Gene_Name,Ywhae
...,...,...,...
2173534,Q4U213,Gene_Name,PTPphi
2173546,Q9CVJ2,Gene_Name,1810058N15Rik
2173581,Q9D449,Gene_Name,Hsfy2
2173596,Q3V1Y1,Gene_Name,4933407I18Rik


In [ ]:
df_mouse = df_mouse.rename(columns={'UniProtID': 'Uniprot_ID'})
df_mouse = df_mouse.rename(columns={'Value': 'Gene_Symbol'})

In [13]:
df_mouse = df_mouse.drop(columns=['Type'])

In [19]:
df_mouse.to_csv("uniprot_gene_mapping_mouse.csv", index=False)

In [4]:
import pandas as pd
import gzip
import urllib.request
import os

# 1. 매핑 함수 정의 (앞서 만든 것과 동일)
def get_uniprot_mapping(url, file_path, target_type, output_col_name, tax_id, organism_name):
    if not os.path.exists(file_path):
        print(f"다운로드 중: {organism_name}...")
        try:
            urllib.request.urlretrieve(url, file_path)
        except Exception as e:
            print(f"❌ 실패: {e}")
            return pd.DataFrame()
    
    print(f"처리 중: {organism_name} ({target_type})...")
    results = []
    try:
        with gzip.open(file_path, 'rt') as f:
            reader = pd.read_csv(f, sep='\t', header=None, names=['UniProtID', 'Type', 'Value'], chunksize=100000)
            for chunk in reader:
                subset = chunk.loc[chunk['Type'] == target_type, ['UniProtID', 'Value']]
                results.append(subset)
    except Exception as e:
        print(f"❌ 읽기 오류: {e}")
        return pd.DataFrame()

    if results:
        df = pd.concat(results, ignore_index=True)
        df.columns = ['uniprot_id', output_col_name]
        df.drop_duplicates(inplace=True)
        
        # [핵심] 구분자 컬럼 추가
        df['tax_id'] = tax_id
        df['organism'] = organism_name
        return df
    return pd.DataFrame()

# ==========================================
# 실행 및 병합 (Merge)
# ==========================================
os.makedirs("uniprot-data", exist_ok=True)

# 1. Human 데이터 가져오기
df_human = get_uniprot_mapping(
    url="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/HUMAN_9606_idmapping.dat.gz",
    file_path="uniprot-data/HUMAN_9606_idmapping.dat.gz",
    target_type='Gene_Name',
    output_col_name='gene_symbol',
    tax_id=9606,
    organism_name='Homo sapiens'
)

# 2. Mouse 데이터 가져오기
df_mouse = get_uniprot_mapping(
    url="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/idmapping/by_organism/MOUSE_10090_idmapping.dat.gz",
    file_path="uniprot-data/MOUSE_10090_idmapping.dat.gz",
    target_type='Gene_Name',
    output_col_name='gene_symbol',
    tax_id=10090,
    organism_name='Mus musculus'
)

# 3. [핵심] 데이터 하나로 합치기 (Concatenate)
print(f"\n 데이터 병합 중...")
# ignore_index=True: 인덱스를 0부터 새로 매깁니다.
df_integrated = pd.concat([df_human, df_mouse], ignore_index=True)

# 4. 결과 확인 및 저장
print(f"{'='*50}")
print(f"최종 통합 데이터 완료")
print(f"{'='*50}")
print(f"총 행 수: {len(df_integrated):,}개")
print(f"컬럼 목록: {list(df_integrated.columns)}")

# 종별 데이터 개수 확인
print("\n[종별 데이터 분포]")
print(df_integrated['organism'].value_counts())

# 파일 저장
OUTPUT_FILE = "uniprot-data/uniprot_data_mouse_human.csv"
df_integrated.to_csv(OUTPUT_FILE, index=False)
print(f"\n저장 완료: {OUTPUT_FILE}")

# 미리보기
print("\n[데이터 미리보기]")
print(df_integrated.head())  # Human 쪽
print(df_integrated.tail())  # Mouse 쪽

다운로드 중: Homo sapiens...
처리 중: Homo sapiens (Gene_Name)...
다운로드 중: Mus musculus...
처리 중: Mus musculus (Gene_Name)...

 데이터 병합 중...
최종 통합 데이터 완료
총 행 수: 248,430개
컬럼 목록: ['uniprot_id', 'gene_symbol', 'tax_id', 'organism']

[종별 데이터 분포]
organism
Homo sapiens    165777
Mus musculus     82653
Name: count, dtype: int64

저장 완료: uniprot-data/uniprot_data_mouse_human.csv

[데이터 미리보기]
  uniprot_id gene_symbol  tax_id      organism
0     P31946       YWHAB    9606  Homo sapiens
1     P62258       YWHAE    9606  Homo sapiens
2     Q04917       YWHAH    9606  Homo sapiens
3     P61981       YWHAG    9606  Homo sapiens
4     P31947         SFN    9606  Homo sapiens
       uniprot_id    gene_symbol  tax_id      organism
248425     Q4U213         PTPphi   10090  Mus musculus
248426     Q9CVJ2  1810058N15Rik   10090  Mus musculus
248427     Q9D449          Hsfy2   10090  Mus musculus
248428     Q3V1Y1  4933407I18Rik   10090  Mus musculus
248429     Q3SXC8         Gm5111   10090  Mus musculus


In [ ]:
import pandas as pd
import gzip
import os

def get_multiple_uniprot_mappings(
    file_path: str, 
    target_types: list,
    organism_name: str = 'HUMAN_9606'
) -> pd.DataFrame:
    """
    UniProt ID 매핑 데이터에서 여러 데이터베이스 타입을 한 번에 필터링하여 반환합니다.
    
    Args:
        file_path (str): 로컬 파일 경로 (gz 포함)
        target_types (list): 필터링할 ID 타입 리스트 (예: ['DrugBank', 'ChEMBL', 'KEGG'])
        organism_name (str): 생물종 이름 (파일명 표시용)
        
    Returns:
        pd.DataFrame: ['uniprot_id', 'database_type', 'database_id'] 컬럼을 가진 데이터프레임
    """
    
    if not os.path.exists(file_path):
        print(f"❌ 파일이 존재하지 않습니다: {file_path}")
        return pd.DataFrame()
    
    print(f"🔄 {organism_name} 데이터 처리 중...")
    print(f"📌 필터링 대상: {', '.join(target_types)}")
    
    chunk_size = 100_000
    results = []

    try:
        with gzip.open(file_path, 'rt') as f:
            reader = pd.read_csv(
                f, 
                sep='\t', 
                header=None, 
                names=['uniprot_id', 'database_type', 'database_id'], 
                chunksize=chunk_size
            )
            
            for chunk in reader:
                # 여러 타입을 한 번에 필터링
                subset = chunk[chunk['database_type'].isin(target_types)]
                if not subset.empty:
                    results.append(subset)
                
    except Exception as e:
        print(f"❌ 파일 읽기 오류: {e}")
        return pd.DataFrame()

    # 결과 병합 및 정리
    if results:
        df_final = pd.concat(results, ignore_index=True)
        df_final.drop_duplicates(inplace=True)
        
        # 각 데이터베이스 타입별 개수 출력
        print(f"\n✅ 처리 완료: 총 {len(df_final):,}개 매핑")
        print("\n[데이터베이스별 매핑 개수]")
        type_counts = df_final['database_type'].value_counts()
        for db_type, count in type_counts.items():
            print(f"  - {db_type}: {count:,}개")
        
        return df_final
    else:
        print("⚠️ 해당 타입의 데이터가 없습니다.")
        return pd.DataFrame()


# ==========================================
# 실행: HUMAN 데이터에서 주요 데이터베이스 추출
# ==========================================

# 추출할 데이터베이스 타입 목록
TARGET_DATABASES = [
    'DrugBank',
    'ChEMBL', 
    'KEGG',
    'Reactome',
    'HGNC',
    'Ensembl',
    'Ensembl_TRS',  # Ensembl Transcript
    'Ensembl_PRO',  # Ensembl Protein
    'Gene_Name'     # 유전자명도 포함
]

# 파일 경로
HUMAN_FILE = "/data/project/shinyoung/DB/processed/Uniprot/HUMAN_9606_idmapping.dat.gz"
OUTPUT_DIR = "/data/project/shinyoung/DB/processed/Uniprot/uniprot-data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 데이터 추출
df_multi = get_multiple_uniprot_mappings(
    file_path=HUMAN_FILE,
    target_types=TARGET_DATABASES,
    organism_name='HUMAN_9606'
)

# 결과 저장
if not df_multi.empty:
    OUTPUT_FILE = f"{OUTPUT_DIR}/uniprot_multi_db_mapping_human.csv"
    df_multi.to_csv(OUTPUT_FILE, index=False)
    print(f"\n💾 저장 완료: {OUTPUT_FILE}")
    
    # 데이터 미리보기
    print("\n[데이터 미리보기]")
    print(df_multi.head(10))
    print(f"\n전체 데이터 shape: {df_multi.shape}")